In [1]:
import os
os.environ["FIFTYONE_API_URI"] = "https://reverse-fashion-api.fiftyone.ai"
os.environ["FIFTYONE_API_KEY"] = "6a845df30ffb0e164ebe93a1|phrOiupiuEItQXfFT1v8KO0zN3Buobqim2IwZUzcJRI"

In [2]:
import fiftyone as fo

/home/sagemaker-user/reverse-fashion-gender-season--training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = fo.load_dataset("sellpy2")
sample = dataset.first()
print("Filepath:", sample.filepath)
print("Media type:", dataset.media_type)
print("Group slices:", dataset.group_slices)

Filepath: s3://reversefashion-images/s/QJs0CvdxmU-NMke6Vkru8-d85c-single.jpg
Media type: group
Group slices: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', 'flatlay']


# ResNet-50 Classifier Training on FiftyOne Sellpy Dataset

Images are stored on S3 (`s3://reversefashion-images/`) and accessible directly from this SageMaker instance — no download needed.

**Configuration:** Set `TARGET_FIELD` to the FiftyOne field path you want to classify (e.g. `"demography"`, `"season"`, `"category_lvl0"`).

In [4]:
# === CONFIGURATION ===
TARGET_FIELD = "category_lvl0"  # FiftyOne field path to classify — change this to your target
BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
VAL_SPLIT = 0.1
GROUP_SLICE = "0"  # Which image slice to use from the group dataset

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from helper import build_dataloaders, build_resnet50, train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
# Load dataset and filter to samples that have the target field populated

view = dataset.match(fo.ViewField("category_lvl1") == 'Clothing')
view = view.match(fo.ViewField(TARGET_FIELD).exists())
view = view.select_group_slices(GROUP_SLICE)
print(f"Samples with '{TARGET_FIELD}' populated: {len(view)}")

# Build class mapping
labels = view.values(TARGET_FIELD)
unique_labels = sorted(set(l for l in labels if l is not None))
class_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
num_classes = len(unique_labels)
print(f"Number of classes: {num_classes}")
print(f"Classes: {unique_labels}")
print(f"Class distribution: {Counter(labels).most_common(10)}")

Samples with 'category_lvl0' populated: 520843
Number of classes: 3
Classes: ['Kids', 'Men', 'Women']
Class distribution: [('Women', 385505), ('Men', 69307), ('Kids', 66031)]


In [7]:
# Bulk fetch filepaths and labels (fast server-side operation)
filepaths = view.values("filepath")
target_values = view.values(TARGET_FIELD)
sample_data = [(fp, class_to_idx[lbl]) for fp, lbl in zip(filepaths, target_values) if lbl in class_to_idx]
print(f"Total usable samples (before balancing): {len(sample_data)}")

# Balance classes by subsampling to the size of the smallest class
import random
random.seed(42)
from collections import defaultdict

samples_by_class = defaultdict(list)
for item in sample_data:
    samples_by_class[item[1]].append(item)

min_count = min(len(v) for v in samples_by_class.values())
sample_data = []
for cls_idx in sorted(samples_by_class):
    sample_data.extend(random.sample(samples_by_class[cls_idx], min_count))

print(f"Samples per class after balancing: {min_count}")
print(f"Total balanced samples: {len(sample_data)}")

Total usable samples (before balancing): 520843
Samples per class after balancing: 66031
Total balanced samples: 198093


In [8]:
train_loader, val_loader, train_size, val_size = build_dataloaders(
    sample_data, VAL_SPLIT, IMAGE_SIZE, BATCH_SIZE,
)
print(f"Train samples: {train_size}, Val samples: {val_size}")

Train samples: 178284, Val samples: 19809


In [9]:
model = build_resnet50(num_classes, device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=LEARNING_RATE / 10,
    max_lr=LEARNING_RATE,
    step_size_up=2 * len(train_loader),
    mode="triangular2",
    cycle_momentum=False,
)
print(f"Model: ResNet-50 | Output classes: {num_classes}")
print(f"Cyclic LR: base_lr={LEARNING_RATE/10:.1e}, max_lr={LEARNING_RATE:.1e}, step_size_up={2*len(train_loader)}")

Model: ResNet-50 | Output classes: 3
Cyclic LR: base_lr=1.0e-05, max_lr=1.0e-04, step_size_up=5572


In [10]:
train(model, train_loader, val_loader, criterion, optimizer, scheduler,
     device, NUM_EPOCHS, class_to_idx, TARGET_FIELD)

train first epoch


KeyboardInterrupt: 